# QLoRA Fine-Tuning (Colab)

Fine-tune a small instruct model with **QLoRA** on a free Colab **T4 GPU**, then compare it to the
un-tuned base. The data/eval helpers come from the `lora-finetune-lab` package; training runs here.

**Runtime → Change runtime type → T4 GPU** before running.

In [ ]:
# 1. Install the repo (with the GPU training stack) + dependencies.
!pip -q install "git+https://github.com/Arunops700/lora-finetune-lab.git#egg=lora-finetune-lab[train]"
import torch; print('CUDA available:', torch.cuda.is_available())

In [ ]:
# 2. Build data: synthetic instruction examples (swap in your own JSONL for real tasks).
from lora_finetune_lab.dataset import generate_synthetic, split
from lora_finetune_lab.prompts import format_text

examples = generate_synthetic(400)
train, val = split(examples, val_fraction=0.1)
train_texts = [format_text(ex) for ex in train]
print(len(train), 'train /', len(val), 'val')
print(train_texts[0])

In [ ]:
# 3. Configure and run QLoRA training (loads the base in 4-bit, trains LoRA adapters).
from lora_finetune_lab.config import TrainingConfig
from lora_finetune_lab.train import run_training

cfg = TrainingConfig(base_model='Qwen/Qwen2.5-0.5B-Instruct', epochs=2, output_dir='outputs/adapter')
adapter_dir = run_training(cfg, train_texts)
print('Adapter saved to', adapter_dir)

In [ ]:
# 4. Evaluate base vs fine-tuned on the held-out val set.
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
from lora_finetune_lab.evaluation import accuracy

tok = AutoTokenizer.from_pretrained(cfg.base_model)
base = AutoModelForCausalLM.from_pretrained(cfg.base_model, device_map='auto')
tuned = PeftModel.from_pretrained(base, adapter_dir)

class HFModel:
    def __init__(self, model):
        self.model = model
    def generate(self, prompt):
        ids = tok(prompt, return_tensors='pt').to(self.model.device)
        out = self.model.generate(**ids, max_new_tokens=16, do_sample=False)
        return tok.decode(out[0][ids['input_ids'].shape[1]:], skip_special_tokens=True)

print('base   :', accuracy(HFModel(base), val))
print('tuned  :', accuracy(HFModel(tuned), val))

## Read the result, then decide

If the tuned model beats the base **on a held-out set**, the fine-tune earned its keep. If it
doesn't — or a cheaper prompt/RAG change would have worked — don't ship it. See
[`docs/when-to-finetune.md`](../docs/when-to-finetune.md) for the decision framework.